# MATH-500 token-budget sensitivity screen

The valid float32 pilot screened MATH-500 at 50% accuracy, but 31 of 64 generations hit the 2,048-token ceiling. This notebook changes only that ceiling to 4,096 tokens. It first evaluates the exact same 64 questions, then runs a fresh disjoint 64-question confirmation only if the unchanged 60–85% accuracy and 512-token trace gate is recovered.

This is a diagnostic screen, not the KV-compression experiment. Use a Kaggle **T4** with Internet enabled and **Save Version → Save & Run All**.

## 1. Configure

Attach the dataset `jonraza15/kv-compression-risk-pilot-export-third-run`. Leave `REFERENCE_INPUT` empty for automatic discovery. Attach a prior export from this notebook only when resuming after exit code 42.

In [ ]:
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "main"
REPO_DIR = "/kaggle/working/latent-reasoning"

REFERENCE_INPUT = ""  # Optional path containing the valid third-run pilot.
RESUME_INPUT = ""  # Optional prior outputs/kv_risk_math_token_budget tree.
RUN_PREFLIGHT = True
RUN_TOKEN_BUDGET_SCREEN = True
UPLOAD_AS_KAGGLE_DATASET = False
KAGGLE_DATASET_HANDLE = "jonraza15/kv-risk-math-token-budget"

MAX_SECONDS = 32400
CONFIG = "configs/kv_risk_math_token_budget.yaml"
PREFLIGHT_RELATIVE = "outputs/kv_risk_math_token_budget_preflight"
OUTPUT_RELATIVE = "outputs/kv_risk_math_token_budget"
REPORT_RELATIVE = "reports/kv_risk_math_token_budget"
LOG_RELATIVE = "logs/kv_risk_math_token_budget"

## 2. Clone pinned code and install dependencies

In [ ]:
import datetime
import hashlib
import json
import os
import pathlib
import shutil
import subprocess
import sys
import time

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("Hugging Face authentication: Kaggle secret loaded")
except Exception:
    print("Hugging Face authentication: public access")

repo = pathlib.Path(REPO_DIR)
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
target = f"origin/{RUN_COMMIT}" if RUN_COMMIT == "main" else RUN_COMMIT
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--detach", target], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements-kv-risk-pilot.txt")],
    check=True,
)
os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Checked out:", commit)
if RUN_COMMIT == "main":
    print("PIN RUN_COMMIT BEFORE THE FINAL RUN:", commit)

## 3. Verify the T4 and code contract

Float32 is fixed here because the paired 2,048-token reference was generated in float32 on a T4. Changing precision would confound the token-cap comparison.

In [ ]:
import torch

assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator"
gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
print("Torch:", torch.__version__, "GPU:", gpu_name, "capability:", (major, minor))
assert major >= 7, "Use a T4 or newer GPU. The current PyTorch build does not support P100 reliably."

subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_kv_risk_cache.py",
        "tests/test_kv_risk_pilot.py",
        "tests/test_kv_risk_preflight.py",
        "tests/test_kv_risk_math_token_budget.py",
        "tests/test_kaggle_kv_risk_math_token_budget_notebook.py",
    ],
    cwd=REPO_DIR,
    check=True,
)

## 4. Locate the valid third-run reference and restore an optional partial run

In [ ]:
PREFLIGHT_ROOT = repo / PREFLIGHT_RELATIVE
OUTPUT_ROOT = repo / OUTPUT_RELATIVE
REPORT_ROOT = repo / REPORT_RELATIVE
LOG_ROOT = repo / LOG_RELATIVE
for root in (PREFLIGHT_ROOT, OUTPUT_ROOT, REPORT_ROOT, LOG_ROOT):
    root.mkdir(parents=True, exist_ok=True)

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def discover_reference_root():
    search_root = pathlib.Path(REFERENCE_INPUT) if REFERENCE_INPUT else pathlib.Path("/kaggle/input")
    if REFERENCE_INPUT:
        assert search_root.is_dir(), f"REFERENCE_INPUT does not exist: {search_root}"
    direct = search_root / "screen/math500/full/run_manifest.json"
    manifests = [direct] if direct.is_file() else list(search_root.rglob("kv_compression_risk_pilot/screen/math500/full/run_manifest.json"))
    discovered = sorted({path.parents[3].resolve() for path in manifests}, key=str)
    valid = []
    for root in discovered:
        manifest_path = root / "screen/math500/full/run_manifest.json"
        summary_path = root / "screen/math500/full/summary.json"
        selection_path = root / "screen/dataset_selection.json"
        predictions_path = root / "screen/math500/full/predictions.jsonl"
        records_dir = root / "screen/math500/full/records"
        if not all(path.is_file() for path in (manifest_path, summary_path, selection_path, predictions_path)):
            continue
        manifest = json.loads(manifest_path.read_text())
        summary = json.loads(summary_path.read_text())
        if manifest.get("state") != "complete" or manifest.get("model_dtype") != "torch.float32":
            continue
        if int(manifest.get("max_new_tokens", -1)) != 2048 or int(summary.get("examples", -1)) != 64:
            continue
        if len(list(records_dir.glob("*.json"))) != 64:
            continue
        fingerprint = (
            str(manifest.get("model_revision")),
            str(manifest.get("request_sha256")),
            str(manifest.get("example_sha256")),
            file_sha256(predictions_path),
        )
        valid.append((root, fingerprint))
    assert valid, f"No complete float32 third-run reference found. Discovered: {discovered}"
    fingerprints = {fingerprint for _, fingerprint in valid}
    assert len(fingerprints) == 1, f"Conflicting third-run references found: {valid}"
    roots = [root for root, _ in valid]
    chosen = min(roots, key=lambda root: (len(root.parts), len(str(root)), str(root)))
    if len(roots) > 1:
        print("Equivalent duplicate reference trees found; using:", chosen)
    return chosen

def discover_resume_root():
    if RESUME_INPUT:
        candidate = pathlib.Path(RESUME_INPUT)
        assert candidate.is_dir(), f"RESUME_INPUT does not exist: {candidate}"
        return candidate
    manifests = list(pathlib.Path("/kaggle/input").rglob("kv_risk_math_token_budget/run_manifest.json"))
    roots = sorted({path.parent.resolve() for path in manifests}, key=str)
    if len(roots) > 1:
        raise RuntimeError(f"Multiple resume trees found. Set RESUME_INPUT explicitly: {roots}")
    return roots[0] if roots else None

REFERENCE_ROOT = discover_reference_root()
print("Reference pilot:", REFERENCE_ROOT)
resume_root = discover_resume_root()
if resume_root is None:
    print("No token-budget resume attached; starting a new durable run")
else:
    print("Restoring:", resume_root)
    shutil.copytree(resume_root, OUTPUT_ROOT, dirs_exist_ok=True)
    print("Restored to:", OUTPUT_ROOT)

## 5. Mandatory numerical preflight

In [ ]:
SESSION_NEEDS_RESUME = False
PREFLIGHT_PASSED = False

def run_logged(command, log_name, allowed=(0, 42)):
    log_path = LOG_ROOT / log_name
    print("Starting:", " ".join(map(str, command)))
    print("Persistent log:", log_path)
    with log_path.open("a", encoding="utf-8", buffering=1) as log:
        log.write(f"\n=== {datetime.datetime.now(datetime.timezone.utc).isoformat()} {' '.join(map(str, command))} ===\n")
        process = subprocess.Popen(
            command,
            cwd=REPO_DIR,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        code = process.wait()
        log.flush()
    print("Exit code:", code)
    if code not in allowed:
        raise subprocess.CalledProcessError(code, command)
    return code

if not RUN_PREFLIGHT:
    raise RuntimeError("RUN_PREFLIGHT must remain True")
preflight_report = PREFLIGHT_ROOT / "preflight.json"
preflight_code = run_logged(
    [
        sys.executable, "-u", "scripts/validate_kv_risk_preflight.py",
        "--config", CONFIG,
        "--output", str(preflight_report),
        "--device", "cuda",
        "--parity-examples", "2",
        "--parity-tokens", "64",
        "--gate-examples", "8",
        "--gate-max-new-tokens", "1024",
    ],
    "preflight.log",
    allowed=(0, 4),
)
PREFLIGHT_PASSED = preflight_code == 0
print(json.dumps(json.loads(preflight_report.read_text()), indent=2))
if not PREFLIGHT_PASSED:
    print("PREFLIGHT FAILED: the token-budget diagnostic is blocked")

## 6. Run the paired 4,096-token diagnostic and conditional fresh confirmation

The fresh confirmation runs only if the paired 64-question diagnostic recovers the original eligibility band. Exit code 42 means the atomic records are safe and the exported tree should be attached to a new run.

In [ ]:
REPORT_PATH = REPORT_ROOT / "math500_token_budget_4096.json"
if RUN_TOKEN_BUDGET_SCREEN and PREFLIGHT_PASSED:
    code = run_logged(
        [
            sys.executable, "-u", "scripts/run_kv_risk_math_token_budget.py",
            "--config", CONFIG,
            "--reference-root", str(REFERENCE_ROOT),
            "--output-dir", str(OUTPUT_ROOT),
            "--report", str(REPORT_PATH),
            "--device", "cuda",
            "--max-seconds", str(MAX_SECONDS),
        ],
        "math500_token_budget_4096.log",
        allowed=(0, 42),
    )
    SESSION_NEEDS_RESUME = code == 42
else:
    print("Token-budget screen skipped")

## 7. Inspect the decision

In [ ]:
from IPython.display import JSON, Markdown, display

if REPORT_PATH.is_file():
    report = json.loads(REPORT_PATH.read_text())
    display(Markdown(REPORT_PATH.with_suffix(".md").read_text()))
    display(JSON(report))
else:
    print("Report is deferred until the run completes")

## 8. Export durable output

In [ ]:
EXPORT_ROOT = pathlib.Path("/kaggle/working/kv_risk_math_token_budget_export")
if EXPORT_ROOT.exists():
    shutil.rmtree(EXPORT_ROOT)
export_repo = EXPORT_ROOT / "latent-reasoning"
for source, relative in (
    (PREFLIGHT_ROOT, PREFLIGHT_RELATIVE),
    (OUTPUT_ROOT, OUTPUT_RELATIVE),
    (REPORT_ROOT, REPORT_RELATIVE),
    (LOG_ROOT, LOG_RELATIVE),
):
    if source.exists():
        shutil.copytree(source, export_repo / relative, dirs_exist_ok=True)

important = sorted(
    path for path in export_repo.rglob("*")
    if path.is_file() and path.suffix in {".json", ".jsonl", ".md"}
)
checksums = []
for path in important:
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    checksums.append(f"{digest}  {path.relative_to(EXPORT_ROOT)}")
(EXPORT_ROOT / "SHA256SUMS.txt").write_text("\n".join(checksums) + "\n")

print("Export:", EXPORT_ROOT)
print("Preflight passed:", PREFLIGHT_PASSED)
print("Session needs resume:", SESSION_NEEDS_RESUME)
print("Files:", sum(1 for path in EXPORT_ROOT.rglob("*") if path.is_file()))
if SESSION_NEEDS_RESUME:
    print("Save this version with outputs enabled, create a Kaggle dataset, attach it, and rerun all cells.")
elif REPORT_PATH.is_file():
    print("Terminal decision:", json.loads(REPORT_PATH.read_text())["decision"])

In [ ]:
if UPLOAD_AS_KAGGLE_DATASET:
    import kagglehub
    kagglehub.dataset_upload(
        KAGGLE_DATASET_HANDLE,
        str(EXPORT_ROOT),
        version_notes=f"MATH-500 4096-token diagnostic at commit {commit}; resume={SESSION_NEEDS_RESUME}",
    )
    print("Dataset upload complete:", KAGGLE_DATASET_HANDLE)
else:
    print("Dataset upload disabled. Save Version with outputs enabled.")